# Week 0 Day 3: Data Visualization

CariSurg MedTech Pathways, Healthcare AI track.

For Day 3, I made one histogram and one scatter plot from the cleaned triage dataset. I used Pulse for both plots because it was the column I cleaned for Assignment 2.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

print(f"Python version: {sys.version}")
print(f"pandas version: {pd.__version__}")

## Load the dataset

I loaded the same reduced emergency triage dataset used for the first two assignments.

In [ ]:
candidate_paths = [
    Path("EmergencyTriageDataset_Reduced_Dirty.csv"),
    Path("../data/EmergencyTriageDataset_Reduced_Dirty.csv"),
    Path("../source_materials/week0/EmergencyTriageDataset_Reduced_Dirty.csv"),
    Path("../../../source_materials/week0/EmergencyTriageDataset_Reduced_Dirty.csv"),
    Path("/content/drive/MyDrive/ColabNotebooks/CariSurg_Triage_Test/EmergencyTriageDataset_Reduced_Dirty.csv"),
]

FILE_PATH = next((path for path in candidate_paths if path.exists()), None)
if FILE_PATH is None:
    raise FileNotFoundError("Upload the Week 0 CSV or update FILE_PATH.")

df = pd.read_csv(FILE_PATH)
print(f"Loaded {df.shape[0]} rows and {df.shape[1]} columns from {FILE_PATH}")
df.head()

## Clean the columns needed for plotting

I cleaned Gender, GCS, SBP, Temp, and Pulse using the same approach from the earlier notebooks. For this assignment, Pulse and Age are the columns used in the plots.

In [ ]:
gender_map = {"male": 1, "female": 0, "1": 1, "0": 0}
df["Gender"] = df["Gender"].astype("string").str.strip().str.lower().map(gender_map).astype("Int64")

df["GCS"] = pd.to_numeric(df["GCS"], errors="coerce")
df.loc[(df["GCS"] < 3) | (df["GCS"] > 15), "GCS"] = np.nan
df["GCS"] = df["GCS"].fillna(df["GCS"].median())

df["SBP"] = pd.to_numeric(df["SBP"], errors="coerce")
df.loc[(df["SBP"] < 50) | (df["SBP"] > 250), "SBP"] = np.nan
df["SBP"] = df["SBP"].fillna(df["SBP"].median())

def to_celsius(value):
    """Convert a temperature value to Celsius when it appears to be Fahrenheit.

    Values above 60 are treated as Fahrenheit. Missing values stay missing.
    """
    if pd.isna(value):
        return np.nan

    text = str(value).strip()
    try:
        if text.endswith("C"):
            return float(text[:-1])
        if text.endswith("F"):
            return (float(text[:-1]) - 32) * 5 / 9
        return float(text)
    except ValueError:
        return np.nan

df["Temp"] = df["Temp"].apply(to_celsius)
df.loc[(df["Temp"] < 32) | (df["Temp"] > 43), "Temp"] = np.nan
df["Temp"] = df["Temp"].fillna(round(df["Temp"].median(), 1))

df["pulse"] = pd.to_numeric(df["pulse"], errors="coerce")
df.loc[(df["pulse"] < 20) | (df["pulse"] > 250), "pulse"] = np.nan
df["pulse"] = df["pulse"].fillna(df["pulse"].median())

print("Missing values after cleaning:")
print(df[["Age", "Gender", "GCS", "SBP", "Temp", "pulse"]].isna().sum())

## Plot 1: Pulse histogram

Clinical question: What does the heart rate distribution look like, and how many patients fall below or above the usual adult resting range?

I used shaded areas for values below 60 bpm and above 100 bpm so the abnormal ranges are visible without needing to read the raw table.

In [ ]:
plot_dir = Path("week0/outputs/plots")
plot_dir.mkdir(parents=True, exist_ok=True)

fig, ax = plt.subplots(figsize=(8, 4))

ax.hist(df["pulse"], bins=20, edgecolor="black", color="#66BB6A", alpha=0.8)
ax.axvspan(0, 60, alpha=0.10, color="blue", label="Below 60 bpm")
ax.axvspan(100, 250, alpha=0.10, color="red", label="Above 100 bpm")
ax.axvline(x=60, color="blue", linestyle=":", linewidth=1)
ax.axvline(x=100, color="red", linestyle=":", linewidth=1)

ax.set_title("Pulse Distribution in the Week 0 ED Dataset", fontsize=12)
ax.set_xlabel("Pulse (beats per minute)")
ax.set_ylabel("Number of Patients")
ax.set_xlim(20, 180)
ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig(plot_dir / "day3_pulse_histogram.png", dpi=120, bbox_inches="tight")
plt.show()

low_pulse = (df["pulse"] < 60).sum()
high_pulse = (df["pulse"] > 100).sum()
print(f"Patients below 60 bpm: {low_pulse}")
print(f"Patients above 100 bpm: {high_pulse}")

## Plot 2: Age vs Pulse scatter plot

Clinical question: Is there an obvious relationship between patient age and pulse in this sample?

I used transparency because many patients overlap on the same parts of the plot. Darker areas show where more records are clustered.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

ax.scatter(df["Age"], df["pulse"], alpha=0.25, s=18, color="#FF7043", edgecolors="none")
ax.axhline(y=60, color="blue", linestyle=":", linewidth=1, label="60 bpm")
ax.axhline(y=100, color="red", linestyle=":", linewidth=1, label="100 bpm")

ax.set_title("Age and Pulse in the Week 0 ED Dataset", fontsize=12)
ax.set_xlabel("Age (years)")
ax.set_ylabel("Pulse (beats per minute)")
ax.set_ylim(20, 180)
ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig(plot_dir / "day3_age_vs_pulse.png", dpi=120, bbox_inches="tight")
plt.show()

## Short note

The histogram shows most pulse values around the normal adult resting range, with some patients above 100 bpm. The scatter plot does not show a strong straight-line relationship between Age and Pulse in this sample. The main cluster sits between about 60 and 110 bpm across different ages.